# Practical Task 6: Deploy an ML Model with FastAPI and Docker

This practice follows the assignment in six stages: train and save a classifier, create an API, run it locally, write a Dockerfile, build an image, and run the container. We continue with the Statlog German Credit dataset used in earlier practices. The target is encoded as `0 = Good Risk` and `1 = Bad Risk`.

# 1) Train a Simple Machine Learning Model

## Task 1.1 Load and inspect the dataset

The training script reads a local copy of the dataset so the same project can be trained on a laptop or inside its Docker image. We first confirm the dataset dimensions and target balance.

In [ ]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "train.py").exists():
    project_dir = Path("practise6").resolve()

column_names = [
    "status_checking", "duration_months", "credit_history", "purpose", "credit_amount",
    "savings", "employment_since", "installment_rate", "personal_status_sex", "other_debtors",
    "present_residence", "property", "age", "other_installment_plans", "housing",
    "number_credits", "job", "people_liable", "telephone", "foreign_worker", "credit_risk",
]
df = pd.read_csv(project_dir / "german.data", sep=r"\s+", header=None, names=column_names)
df.shape, df["credit_risk"].value_counts().sort_index()

The source uses `1` for Good Risk and `2` for Bad Risk. The training script maps these to 0 and 1, then uses a stratified 80/20 split. Numeric features are standardized and categorical features are one-hot encoded inside a pipeline, so the transformations are fitted on training data and saved with the classifier.

## Task 1.2 Train and save the model

In [ ]:
# Run from the project folder to create model.joblib and training_metrics.json.
import os
os.chdir(project_dir)
%run train.py

The saved `model.joblib` is the trained pipeline artifact. The test metrics provide a small held-out check; they do not establish that this historical dataset is appropriate for a real lending decision.

# 2) Create the FastAPI Application

The API loads `model.joblib` when it starts. `GET /` confirms the service is available, while `POST /predict` accepts all 20 input features and returns the predicted class and its readable label. FastAPI validates the request body and generates an interactive schema for Swagger.

In [ ]:
# Example request body for POST /predict. The complete schema is also shown in /docs.
request_payload = {
    "status_checking": "A11", "duration_months": 6, "credit_history": "A34",
    "purpose": "A43", "credit_amount": 1169, "savings": "A65",
    "employment_since": "A75", "installment_rate": 4, "personal_status_sex": "A93",
    "other_debtors": "A101", "present_residence": 4, "property": "A121",
    "age": 67, "other_installment_plans": "A143", "housing": "A152",
    "number_credits": 2, "job": "A173", "people_liable": 1,
    "telephone": "A192", "foreign_worker": "A201",
}
len(request_payload)

# 3) Run and Check the API Locally

Install the dependencies and train the model before starting Uvicorn. Run these commands from the `practise6` directory in a terminal:

```bash
pip install -r requirements.txt
python train.py
uvicorn main:app --reload
```

Open `http://127.0.0.1:8000` for the health response and `http://127.0.0.1:8000/docs` for the interactive documentation. In Swagger, expand `POST /predict`, choose **Try it out**, submit the example request, and check for a response containing `prediction` and `risk_label`.

## Task 3.1 Test the endpoints from a client

With the local API still running, these commands check the root route, make a prediction, and fetch the Swagger page. A successful prediction returns HTTP 200 and a Good Risk or Bad Risk label.

In [ ]:
# Run in a notebook cell while Uvicorn is running.
import json
import urllib.request

with urllib.request.urlopen("http://127.0.0.1:8000/") as response:
    print("GET /:", json.load(response))

request = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(request_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request) as response:
    print("POST /predict:", json.load(response))

with urllib.request.urlopen("http://127.0.0.1:8000/docs") as response:
    print("GET /docs status:", response.status)

# 4) Create a Dockerfile

The Dockerfile uses Python 3.11, installs the pinned dependencies, copies the source and dataset into the image, and trains the model there. The `.dockerignore` excludes the local model artifact, so the container creates `model.joblib` with the same scikit-learn version it uses to serve predictions. Uvicorn then starts on the container's port 8000.

In [ ]:
print((project_dir / "Dockerfile").read_text())

# 5) Build the Docker Image

Build the image from the project directory with `docker build -t german-credit-api .`. The Dockerfile installs dependencies and trains the model during the build, making the image self-contained. The README contains the full copy-paste commands.

# 6) Run and Check the Container

Run the image with `docker run --rm -p 8000:8000 german-credit-api`. The left port is on the host; the right port is inside the container. If the host's port 8000 is occupied, map a different host port, for example `-p 8001:8000`.

Open `/docs` on the mapped host port and verify `GET /` and `POST /predict`. For this run, Swagger opened successfully, the root endpoint returned `{"message": "ML API is running"}`, and the example prediction returned class `0` (`Good Risk`).

# 7) Summary

This project packages the full preprocessing and Logistic Regression pipeline as a reusable model artifact and exposes predictions through FastAPI. The API has a health route and a validated prediction route, and Docker provides a repeatable runtime. Before submission, keep the notebook and project files together and add the project to a Git repository so its Git link can be submitted. The model is a learning example, not a production credit-scoring system.